# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, following best practices in referencing entities by their Croissant `@id`.

### Dataset Source
The dataset is defined by a Croissant schema and accessed via this URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the metadata object directly
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in Croissant datasets are referenced by their `@id`. Here, we'll inspect the available record sets, fields and columns (if present) by their `@id`.

In [ ]:
# List all record sets and their @id
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')})")

# Review fields and columns for each record set
for rs in record_sets:
    print(f"\nFields for record set {rs['@id']}:")
    for field in rs.get('fields', []):
        print(f"  - Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
        # Show columns if defined
        if 'columns' in field:
            print("    Columns:")
            for col in field['columns']:
                print(f"      - Column @id: {col['@id']}, name: {col.get('name', 'N/A')}, dataType: {col.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.
Use the record set and field `@id`s from the overview. All operations reference dataset elements by their unique `@id`.

In [ ]:
# Extract data from each record set, using @id for reference
dataframes = {}

record_set_ids = [rs['@id'] for rs in record_sets]
print("Loading data for record sets:@id:")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"- {record_set_id}: shape {df.shape}, columns: {df.columns.tolist()}\n")

# Display first few rows of the first record set
first_record_set_id = record_set_ids[0] if record_set_ids else None
if first_record_set_id:
    print(f"Sample records from record set {first_record_set_id}:")
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations reference fields by their `@id` and work with DataFrames loaded above.

We'll demonstrate filtering, normalization, and grouping on numeric and categorical fields.

In [ ]:
# Example: identify a numeric and group field by @id for demonstration

# Choose one record set for EDA
record_set_id = first_record_set_id
df = dataframes.get(record_set_id)

# Attempt to identify numeric fields via metadata
numeric_field_id = None
group_field_id = None
fields = None

for rs in record_sets:
    if rs['@id'] == record_set_id:
        fields = rs.get('fields', [])
        for field in fields:
            if field.get('dataType', '').lower() in ['integer', 'float', 'number']:
                numeric_field_id = field['@id']
            if field.get('dataType', '').lower() in ['text', 'string'] and group_field_id is None:
                group_field_id = field['@id']
        break

print(f"Numeric field @id: {numeric_field_id}")
print(f"Group field @id: {group_field_id}")

# Provide explicit fallback if not found in metadata
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for this record set; please check data or update field @id.")

## 5. Visualization
Visualize the distribution of a numeric variable and relationship to a categorical variable.

Here we'll plot a histogram and boxplot if numeric and group fields are present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group_field if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated loading the FAIR^2 dataset defined by a Croissant schema, reviewing its record sets by their `@id`, extracting data, and carrying out basic processing and visualization referencing record sets, fields, and columns by their unique `@id`.

You can extend this workflow by referencing other entities by their `@id`, further processing multiple record sets, or integrating other analysis and visualization techniques as required.